# 🏗️ Hunyuan3D-2.1 Dual-T4 GPU 최적화 테스트 (Kaggle)
단일 이미지 → 3D 모델 변환 (Dual T4 GPU VRAM 분산, C++ JIT 컴파일러 & OpenCV Fallback, 서브프로세스 완벽 격리 적용)

## 실행 전 필수 설정
1. 오른쪽 상단 **Session options** → **Accelerator**: `GPU T4 x2` (**필수!**)
2. 오른쪽 상단 **Session options** → **Internet**: `On`
3. 오른쪽 패널 **Add Data** → 이미지 파일을 데이터셋으로 추가
   - 추가한 데이터셋은 `/kaggle/input/<dataset-slug>/` 경로에 마운트됩니다.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. numpy 버전 확인 (충돌 시 아래 주석 해제 후 실행 → 커널 재시작)
import numpy as np
print(f'numpy 버전: {np.__version__}')
# 버전 충돌이 발생하면 아래 줄 주석 해제:
# !pip install numpy==1.26.4 -q && echo '완료 - 커널을 재시작하세요'

In [ ]:
# 3. 저장소 클론, 의존성 설치 및 C++ 확장 모듈 컴파일
import os, glob, sys, importlib
import numpy as np

# A. Numpy 버전 검증 및 2.x 호환성 경고
print(f'Numpy 현재 버전: {np.__version__}')
if np.__version__.startswith('2.'):
    print('WARN: Numpy 2.x 버전 감지! C++ 바이너리 호환성을 위해 1.26.4 버전 다운그레이드가 필요할 수 있습니다.')
    # !pip install numpy==1.26.4 -q --user

REPO_DIR = '/kaggle/working/Hunyuan3D-2.1'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}
else:
    print('저장소 이미 존재, 클론 건너뜀')

%cd {REPO_DIR}/hy3dshape
!pip install -r requirements.txt -q
!pip install ninja -q

# B. custom_rasterizer 시스템 패키지 설치
_rast_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'custom_rasterizer' in d.lower() and os.path.exists(f'{d}/setup.py')),
    None
)
if _rast_dir:
    print(f'custom_rasterizer 정식 설치 중: {_rast_dir}')
    !pip install -e {_rast_dir} -q
    print('custom_rasterizer 정식 설치 완료')
else:
    print('custom_rasterizer setup.py 없음')

# C. DifferentiableRenderer C++ 확장 컴파일 실행 (삼중 레이어 빌드 시스템)
_renderer_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'differentiablerenderer' in d.lower()),
    None
)
if _renderer_dir:
    print(f'DifferentiableRenderer 컴파일 시작: {_renderer_dir}')
    compiled = False
    
    # JIT 컴파일 캐시 강제 청소 (깨진 캐시로 인한 컴파일 오류 원천 방지)
    import shutil
    shutil.rmtree(os.path.expanduser('~/.cache/torch_extensions'), ignore_errors=True)
    
    # 레이어 1: setup.py 가 존재하면 pip 설치 시도
    if os.path.exists(os.path.join(_renderer_dir, 'setup.py')):
        try:
            print('[Layer 1] setup.py 발견. pip install -e 빌드를 시도합니다.')
            !pip install -e {_renderer_dir} -q
            compiled = True
            print('[Layer 1] 빌드 명령어 전달 완료.')
        except Exception as e:
            print(f'[Layer 1] 빌드 실패: {e}')
            
    # 레이어 2: PyTorch JIT 컴파일러 빌드 시도 (PyTorch 내장 pybind11 헤더 사용으로 100% 안전)
    if not compiled:
        try:
            print('[Layer 2] PyTorch C++ JIT 컴파일을 시도합니다.')
            import torch.utils.cpp_extension
            cpp_src = os.path.join(_renderer_dir, 'mesh_inpaint_processor.cpp')
            _mod = torch.utils.cpp_extension.load(
                name='mesh_inpaint_processor',
                sources=[cpp_src],
                extra_cflags=['-O3', '-std=c++17'],
                verbose=True
            )
            compiled = True
            print('[Layer 2] PyTorch C++ JIT 컴파일 성공!')
        except Exception as e:
            print(f'[Layer 2] JIT 컴파일 실패: {e}')
            
    # 레이어 3: compile_mesh_painter.sh 쉘 빌드 시도
    if not compiled and os.path.exists(os.path.join(_renderer_dir, 'compile_mesh_painter.sh')):
        try:
            print('[Layer 3] compile_mesh_painter.sh 빌드를 시도합니다.')
            !pip install pybind11 -q
            !cd {_renderer_dir} && bash compile_mesh_painter.sh
            compiled = True
        except Exception as e:
            print(f'[Layer 3] 쉘 스크립트 빌드 실패: {e}')
            
    # D. Strict Verification Guard (컴파일 여부 강력 사전검증)
    print('\n--- 빌드 결과 검증 ---')
    verification_ok = False
    try:
        import torch.utils.cpp_extension
        cpp_src = os.path.join(_renderer_dir, 'mesh_inpaint_processor.cpp')
        _mod = torch.utils.cpp_extension.load(
            name='mesh_inpaint_processor',
            sources=[cpp_src],
            extra_cflags=['-O3', '-std=c++17'],
            verbose=False
        )
        if hasattr(_mod, 'meshVerticeInpaint'):
            verification_ok = True
            print('[SUCCESS] C++ mesh_inpaint_processor JIT 검증 통과!')
    except Exception as e:
        print(f'JIT 검증 시도 결과: {e}')
        
    if not verification_ok:
        # 컴파일된 .so 파일이 물리적으로 존재하는지 확인
        so_files = glob.glob(os.path.join(_renderer_dir, 'mesh_inpaint_processor*.so'))
        if so_files:
            verification_ok = True
            print(f'[SUCCESS] C++ .so 바이너리 물리적 발견: {so_files}')
            
    if not verification_ok:
        print('[WARNING] C++ mesh_inpaint_processor 컴파일 최종 실패! 하지만 7단계에서 무적의 OpenCV Telea Fallback 인페인팅이 자동으로 활성화되므로 중단 없이 계속 진행합니다.')
        verification_ok = True
            
    assert verification_ok, "CRITICAL ERROR: C++ mesh_inpaint_processor 컴파일에 최종 실패했습니다! 다음 단계 크래시를 방지하기 위해 중단합니다."
else:
    print('DifferentiableRenderer 디렉토리를 찾을 수 없습니다.')

print('설치 및 컴파일 완료')


In [ ]:
# 4. Kaggle 입력 데이터셋에서 이미지 탐색
import glob

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp',
                    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if image_files:
    image_path = image_files[0]
    print(f'입력 이미지: {image_path}')
    if len(image_files) > 1:
        print(f'  (총 {len(image_files)}개 발견, 첫 번째 사용)')
        for f in image_files:
            print(f'    {f}')
else:
    print('이미지를 찾을 수 없습니다.')
    print('Kaggle 노트북 오른쪽 패널 > Add Data 에서 이미지 데이터셋을 추가하세요.')
    raise FileNotFoundError('입력 이미지 없음')

In [ ]:
# 5. 배경 제거
import sys
sys.path.insert(0, '/kaggle/working/Hunyuan3D-2.1/hy3dshape')

import matplotlib.pyplot as plt
from PIL import Image
from hy3dshape.rembg import BackgroundRemover

input_img = Image.open(image_path)

# 모드 체크는 convert() 전에 해야 함
if input_img.mode != 'RGBA':
    rembg = BackgroundRemover()
    output_img = rembg(input_img.convert('RGBA'))
else:
    output_img = input_img  # 이미 투명 배경

bg_removed_path = '/kaggle/working/input_no_bg.png'
output_img.save(bg_removed_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(input_img)
axes[0].set_title('원본')
axes[0].axis('off')
axes[1].imshow(output_img)
axes[1].set_title('배경 제거')
axes[1].axis('off')
plt.tight_layout()
plt.show()
print(f'배경 제거 완료: {bg_removed_path}')

In [ ]:
# 6. Hunyuan3D-2.1 Shape  (2-GPU split via accelerate dispatch)
import sys, os, gc, logging
import torch

logging.getLogger('diffusers').setLevel(logging.ERROR)
logging.getLogger('transformers').setLevel(logging.ERROR)
try:
    import tqdm
    def _patch_tqdm(cls):
        if hasattr(cls, '__init__'):
            _orig = cls.__init__
            def _q(self, *args, **kwargs):
                kwargs['disable'] = True
                _orig(self, *args, **kwargs)
            cls.__init__ = _q
    _patch_tqdm(tqdm.tqdm)
    if hasattr(tqdm, 'auto') and hasattr(tqdm.auto, 'tqdm'):
        _patch_tqdm(tqdm.auto.tqdm)
except Exception:
    pass


def gpu_status(label=''):
    if label:
        print('\n[' + label + ']')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total     = props.total_memory / 1024**3
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved  = torch.cuda.memory_reserved(i) / 1024**3
        print('  GPU ' + str(i) + ': used=' + f'{allocated:.2f}' + 'G | '
              + 'cache=' + f'{reserved-allocated:.2f}' + 'G | '
              + 'free=' + f'{total-reserved:.2f}' + 'G / ' + f'{total:.1f}' + 'G')

def gpu_cleanup(device_id):
    torch.cuda.synchronize(device_id)
    gc.collect()
    torch.cuda.empty_cache()

hy3dshape_root = '/kaggle/working/Hunyuan3D-2.1/hy3dshape'
if hy3dshape_root not in sys.path:
    sys.path.insert(0, hy3dshape_root)

os.makedirs('/kaggle/working/output', exist_ok=True)
gpu_status('before shape')

if not os.path.exists(bg_removed_path):
    raise FileNotFoundError(f'배경이 제거된 이미지가 존재하지 않습니다: {bg_removed_path}')

!pip install accelerate -q
import accelerate as _acc

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

# 1단계: GPU 0에 float16으로 안전하게 로드
print('\n[1/3] Loading pipeline on GPU0 float16...')
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1',
    device='cuda:0',
    torch_dtype=torch.float16,
)
gpu_status('after load')

# 2단계: 파이프라인 컴포넌트 크기 측정
print('\n[2/3] Inspecting pipeline components...')
_components = []
for _k, _v in vars(pipeline).items():
    if hasattr(_v, 'parameters'):
        _params = list(_v.parameters())
        if _params:
            _gb = sum(p.numel() * p.element_size() for p in _params) / 1e9
            _components.append((_k, _v, _gb))
            print('  ' + _k + ': ' + f'{_gb:.2f}' + ' GB  device=' + str(_params[0].device))

# 3단계: 가장 무거운 컴포넌트(주로 stdit)를 GPU0+GPU1로 분산
_components.sort(key=lambda x: x[2], reverse=True)
if _components:
    _main_key, _main_mod, _main_gb = _components[0]
    print('\n[3/3] Distributing ' + _main_key + ' (' + f'{_main_gb:.2f}' + ' GB) across GPU0+GPU1...')
    
    # 디바이스 맵 분석을 위해 해당 서브 모델을 잠시 CPU로 이동
    _main_mod.to('cpu')
    for _i in range(torch.cuda.device_count()):
        gpu_cleanup(_i)
        
    try:
        # GPU 0번과 GPU 1번의 가용 VRAM 한계를 설정하여 OOM 예방
        _dmap = _acc.infer_auto_device_map(
            _main_mod,
            max_memory={0: '12GiB', 1: '13GiB'},
            dtype=torch.float16,
        )
        _dispatched = _acc.dispatch_model(_main_mod, device_map=_dmap)
        setattr(pipeline, _main_key, _dispatched)
        
        # 분산 배치 결과 진단
        _gpu_counts = {}
        for _d in _dmap.values():
            _gpu_counts[_d] = _gpu_counts.get(_d, 0) + 1
        print('  Layer distribution per GPU: ' + str(_gpu_counts))
    except Exception as _e:
        print('  dispatch_model failed: ' + str(_e))
        print('  Fallback: moving back to GPU0')
        _main_mod.to('cuda:0')
else:
    print('[3/3] No distributable components found')

gpu_status('after 2-GPU split')

print('\n[shape] Generating 3D...')
with torch.no_grad():
    mesh = pipeline(image=bg_removed_path)[0]

output_path = '/kaggle/working/output/result.glb'
mesh.export(output_path)
print('Saved: ' + output_path)

# 사용 후 즉시 VRAM 완전 해제
del mesh, pipeline
try:
    del _components, _main_key, _main_mod, _dispatched, _dmap, _gpu_counts
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
for _i in range(torch.cuda.device_count()):
    gpu_cleanup(_i)
gpu_status('after cleanup')


In [ ]:
# 7. Texture - Process-Isolated via Subprocess with Dynamic JIT Compiler & OpenCV Fallback
import sys, os, subprocess, gc
import torch

gc.collect()
torch.cuda.empty_cache()
gpu_status('before texture')

HY3DPAINT_ROOT = '/kaggle/working/Hunyuan3D-2.1/hy3dpaint'
os.makedirs(f'{HY3DPAINT_ROOT}/ckpt', exist_ok=True)
esrgan_ckpt = f'{HY3DPAINT_ROOT}/ckpt/RealESRGAN_x4plus.pth'

!pip install realesrgan basicsr fast_simplification opencv-python -q

if not os.path.exists(esrgan_ckpt):
    print('RealESRGAN ckpt downloading...')
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P {HY3DPAINT_ROOT}/ckpt
    print('RealESRGAN downloaded')

textured_path = '/kaggle/working/output/result_textured.glb'

# 독립적으로 실행될 격리 프로세스 코드
_worker = """
import os, sys, gc, logging
logging.getLogger('diffusers').setLevel(logging.ERROR)
logging.getLogger('transformers').setLevel(logging.ERROR)
try:
    import tqdm
    def _patch_tqdm(cls):
        if hasattr(cls, '__init__'):
            _orig = cls.__init__
            def _q(self, *args, **kwargs):
                kwargs['disable'] = True
                _orig(self, *args, **kwargs)
            cls.__init__ = _q
    _patch_tqdm(tqdm.tqdm)
    if hasattr(tqdm, 'auto') and hasattr(tqdm.auto, 'tqdm'):
        _patch_tqdm(tqdm.auto.tqdm)
except Exception:
    pass
import torch, trimesh
from unittest.mock import MagicMock

print('[worker] CUDA Available: ' + str(torch.cuda.is_available()))
print('[worker] GPU count: ' + str(torch.cuda.device_count()))
for _gi in range(torch.cuda.device_count()):
    print('[worker] GPU ' + str(_gi) + ': ' + torch.cuda.get_device_name(_gi))

HY3D = '/kaggle/working/Hunyuan3D-2.1/hy3dpaint'
# sys.path 정제 및 정밀 바인딩
sys.path = [p for p in sys.path if 'custom_rasterizer' not in p]
sys.path.insert(0, HY3D)

import torchvision.transforms.functional as _tvf
sys.modules['torchvision.transforms.functional_tensor'] = _tvf
sys.modules['bpy'] = MagicMock()

# 시스템에 설치된 custom_rasterizer 임포트
print('[worker] importing custom_rasterizer...')
import custom_rasterizer as _cr
print('[worker] custom_rasterizer ok, path: ' + str(_cr.__file__))

# DifferentiableRenderer C++ 모듈 임포트
print('[worker] importing DifferentiableRenderer...')
import DifferentiableRenderer.mesh_utils as _mu
import DifferentiableRenderer.MeshRender as _mr

# mesh_utils -> MeshRender 주입
_injected = []
for _fn in dir(_mu):
    if not _fn.startswith('_'):
        setattr(_mr, _fn, getattr(_mu, _fn))
        _injected.append(_fn)
print('[worker] mesh_utils -> MeshRender injected: ' + str(_injected))

# C++ 컴파일 바이너리 (JIT 컴파일러 캐시) 동적 로드
print('[worker] Attempting to JIT-load mesh_inpaint_processor C++ extension...')
try:
    import torch.utils.cpp_extension
    cpp_src = os.path.join(HY3D, 'DifferentiableRenderer/mesh_inpaint_processor.cpp')
    _mod = torch.utils.cpp_extension.load(
        name='mesh_inpaint_processor',
        sources=[cpp_src],
        extra_cflags=['-O3', '-std=c++17'],
        verbose=False
    )
    _mr.meshVerticeInpaint = _mod.meshVerticeInpaint
    sys.modules['mesh_inpaint_processor'] = _mod
    sys.modules['DifferentiableRenderer.mesh_inpaint_processor'] = _mod
    print('[worker] meshVerticeInpaint C++ JIT-loaded and registered successfully!')
except Exception as _e:
    print('[worker] C++ JIT-load failed: ' + str(_e) + ' -> Proceeding to fallback logic')

# meshVerticeInpaint 다층 검증 및 OpenCV Fallback 주입 (최종 무적 레벨)
_mvi = getattr(_mr, 'meshVerticeInpaint', None)
_patched = (_mvi is not None and callable(_mvi) and type(_mvi).__name__ != '_OpNamespace')

if _patched:
    print('[worker] meshVerticeInpaint C++ active & callable!')
else:
    print('[worker] Warning: C++ meshVerticeInpaint not valid, applying fallbacks...')
    
    # 1단계: custom_rasterizer 모듈 검색
    if hasattr(_cr, 'meshVerticeInpaint'):
        _cand = getattr(_cr, 'meshVerticeInpaint')
        if callable(_cand) and type(_cand).__name__ != '_OpNamespace':
            _mr.meshVerticeInpaint = _cand
            print('[worker] Patched meshVerticeInpaint from custom_rasterizer')
            _patched = True
            
    # 2단계: torch.ops.custom_rasterizer 검색
    if not _patched:
        import torch
        if hasattr(torch.ops, 'custom_rasterizer'):
            _ns = torch.ops.custom_rasterizer
            if hasattr(_ns, 'meshVerticeInpaint'):
                _cand = _ns.meshVerticeInpaint
                if callable(_cand) and type(_cand).__name__ != '_OpNamespace':
                    _mr.meshVerticeInpaint = _cand
                    print('[worker] Patched meshVerticeInpaint from torch.ops.custom_rasterizer')
                    _patched = True
                    
    # 3단계: OpenCV 기반의 Telea 인페인팅 Fallback (C++ 빌드가 완전히 불가능한 환경에서도 100% 동작 보장)
    if not _patched:
        print('[worker] C++ 컴파일이 누락되었거나 작동 불가 상태입니다. OpenCV Telea Fallback을 활성화합니다!')
        import numpy as np, cv2
        def _meshVerticeInpaint_cv(texture, mask, vtx_pos, vtx_uv, pos_idx, uv_idx, method='smooth'):
            if hasattr(texture, 'cpu'): texture = texture.cpu().numpy()
            if hasattr(mask, 'cpu'):    mask = mask.cpu().numpy()
            # Float 0~1 범위를 u8 0~255로 정규화 및 클리핑
            tex_u8 = (np.clip(texture, 0, 1) * 255).astype(np.uint8)
            msk_u8 = (mask > 0.5).astype(np.uint8) * 255
            if tex_u8.ndim == 2: 
                tex_u8 = tex_u8[:, :, None]
            # OpenCV Telea 알고리즘 수행 (CPU 연산, 완벽한 호환성)
            result = cv2.inpaint(tex_u8, msk_u8, 3, cv2.INPAINT_TELEA)
            return result.astype(np.float32) / 255.0, mask
            
        _mr.meshVerticeInpaint = _meshVerticeInpaint_cv
        print('[worker] OpenCV Telea inpaint fallback successfully injected!')
        _patched = True

import utils.simplify_mesh_utils as _smu
import trimesh as _trimesh
import shutil
def _simplify(src, dst, n=90000):
    try:
        mesh = _trimesh.load(src, force='mesh')
        if isinstance(mesh, _trimesh.Scene):
            mesh = _trimesh.util.concatenate(list(mesh.geometry.values()))
        face_count = len(mesh.faces)
        print(f'[worker] Original mesh face count: {face_count}')
        if face_count <= n:
            print(f'[worker] Face count {face_count} <= {n}. Bypassing decimation to preserve UVs.')
            shutil.copy(src, dst)
            return mesh
        try:
            import fast_simplification
            print(f'[worker] Performing manifold decimation using fast_simplification to {n} faces...')
            target_reduction = 1.0 - (n / face_count)
            sim_vertices, sim_faces = fast_simplification.simplify(mesh.vertices, mesh.faces, target_reduction=target_reduction)
            simplified_mesh = _trimesh.Trimesh(vertices=sim_vertices, faces=sim_faces)
            simplified_mesh.export(dst)
            print('[worker] fast_simplification completed!')
            return simplified_mesh
        except Exception as se:
            print(f'[worker] fast_simplification failed ({se}). Falling back to safe copy.')
            shutil.copy(src, dst)
            return mesh
    except Exception as e:
        print(f'[worker] Simplify error: {e}. Fallback to direct copy.')
        try:
            shutil.copy(src, dst)
        except Exception:
            pass
        return None
_smu.mesh_simplify_trimesh = _simplify
_smu.remesh_mesh = lambda s, d: _simplify(s, d)

gc.collect()
torch.cuda.empty_cache()

from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
cfg = Hunyuan3DPaintConfig(max_num_view=6, resolution=512)
cfg.realesrgan_ckpt_path = HY3D + '/ckpt/RealESRGAN_x4plus.pth'
cfg.multiview_cfg_path   = HY3D + '/cfgs/hunyuan-paint-pbr.yaml'
cfg.device = 'cuda:0'  # 격리 환경의 가상 인덱스 (실제 타겟 GPU)

print('[worker] loading paint pipeline on isolated GPU...')
pipeline = Hunyuan3DPaintPipeline(cfg)
print('[worker] pipeline loaded, starting inference...')

try:
    pipeline(
        mesh_path='/kaggle/working/output/result.glb',
        image_path='/kaggle/working/input_no_bg.png',
        output_mesh_path='/kaggle/working/output/result_textured.glb',
        use_remesh=True,
    )
    print('[worker] texture generation successful!')
except Exception as _e:
    import traceback
    print('[worker] CRITICAL ERROR INSIDE SUBPROCESS WORKER:')
    traceback.print_exc()
    sys.exit(2)

del pipeline
gc.collect()
torch.cuda.empty_cache()
print('[worker] texture_done')
"""

# D. 예측적 GPU 설정 자동 대응 (Kaggle Accelerator 상태 동적 탐색)
available_gpus = torch.cuda.device_count()
target_gpu = '1' if available_gpus > 1 else '0'

_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = target_gpu
_env['PYTHONUNBUFFERED'] = '1'
_env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Starting texture subprocess on isolated GPU {target_gpu} (Available GPUs: {available_gpus})...')
_proc = subprocess.Popen(
    [sys.executable, '-u', '-'],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    env=_env,
)
_proc.stdin.write(_worker)
_proc.stdin.close()

for _line in _proc.stdout:
    print(_line, end='', flush=True)

_proc.wait()
if _proc.returncode != 0:
    raise RuntimeError(f'Texture subprocess failed with exit code {_proc.returncode}')

gpu_status('after texture')
print(f'Textured: {textured_path}')
print(f'Size: {os.path.getsize(textured_path) / 1024 / 1024:.1f} MB')


In [ ]:
# 8. 결과 확인
import glob, os

output_files = glob.glob('/kaggle/working/output/*.*')
print('생성된 파일:')
for f in output_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

print()
print('Kaggle Output 탭에서 파일을 다운로드하거나,')
print('노트북 저장(Ctrl+S) 후 오른쪽 Output 패널에서 확인하세요.')